# Predictive_Maintenance_and_Operational_Efficiency_in_Wind_Energy

The world energy demand is increasing slowly and this has created the necessity of sustainable and environmental friendly power production mechanisms. Wind energy is one of the other options of renewable energy that has emerged as one of the most developed and commercially viable options. This energy requires wind turbines and due to its complex mechanical attributes and exposure to harsh climate factors often leads to failure of components. Such failures may affect the working efficiency and increase the maintenance expenses.

Conventional maintenance approaches such as reactive maintenance, where problems are resolved after their occurrence and preventive maintenance, where maintenance is conducted according to a timetable are either too expensive or fail to accurately forecast possible
problems. Conversely, predictive maintenance is based on sensor technology, data analytics and machine learning that predict equipment failure before it occurs. This type of preventive approach provide the operators the opportunity to perform maintenance only when required, which minimizes downtime and maximizes reliability of the system.

Renewable energy sources play an increasingly important role in the global energy mix, as the effort to reduce the environmental impact of energy production increases.

Out of all the renewable energy alternatives, wind energy is one of the most developed technologies worldwide. The U.S Department of Energy has put together a guide to achieving operational efficiency using predictive maintenance practices.

Predictive maintenance uses sensor information and analysis methods to measure and predict degradation and future component capability. The idea behind predictive maintenance is that failure patterns are predictable and if component failure can be predicted accurately and the component is replaced before it fails, the costs of operation and maintenance will be much lower.

The sensors fitted across different machines involved in the process of energy generation collect data related to various environmental factors (temperature, humidity, wind speed, etc.) and additional features related to various parts of the wind turbine (gearbox, tower, blades, break, etc.).

## Objective

A company working on improving the machinery/processes involved in the production of wind energy using machine learning and has collected data of generator failure of wind turbines using sensors. They have shared a ciphered version of the data, as the data collected through sensors is confidential (the type of data collected varies with companies). Data has 40 predictors, 40000 observations in the training set and 10000 in the test set.

The objective is to build various classification models, tune them, and find the best one that will help identify failures so that the generator could be repaired before failing/breaking to reduce the maintenance cost. The different costs associated with maintenance are as follows:

- `Replacement cost = $40,000`
- `Repair cost = $15,000`
- `Inspection cost = $5,000`

“1” in the target variables should be considered as “failure” and “0” will represent “No failure”.

## Data Description

- The data provided is a transformed version of original data which was collected using sensors.
- Train.csv - To be used for training and tuning of models.
- Test.csv - To be used only for testing the performance of the final best model.
- Both the datasets consist of 40 predictor variables and 1 target variable

**Importing Liraries**

In [1]:
# To help with reading and manipulating data
import pandas as pd
import numpy as np

# To help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# To be used for missing value imputation
from sklearn.impute import SimpleImputer

# To help with model building
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    BaggingClassifier,
)
import xgboost as xgb
from xgboost import XGBClassifier

# To get different metric scores, and split data
from sklearn import metrics
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score


from sklearn.metrics import(
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,

)

# To be used for data scaling and one hot encoding
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

# To be used for tuning the model
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# To oversample and undersample data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# To be used for creating pipelines and personalizing them
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# To define maximum number of columns to be displayed in a dataframe
pd.set_option("display.max_columns", None)

# To supress scientific notations for a dataframe
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# To supress warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

Loading Data [ Train and Test]

In [ ]:
# Read train data
data = pd.read_csv('/content/drive/MyDrive/Train.csv')

# Make a copy of train data
df=data.copy()

In [ ]:
# Check the dimensions of train data
df.shape

In [ ]:
# Read test data
data_test = pd.read_csv("/content/drive/MyDrive/Test.csv")

# Make a copy of test data

df_test=data_test.copy()

In [ ]:
# Check the dimensions of test data
df_test.shape

## Data Overview

### Viewing the first and last few rows of the dataset

In [ ]:
# let's view the first 5 rows of the data
df.head()

In [ ]:
# let's view the last 5 rows of the data
df.tail()

### Exploring the different datatypes in various columns

In [ ]:
df.info()

- 40 variables are of type float and only the `Target` variable is of type int
- There are some missing values in columns `V1` and `V2`

### Check for missing values in Train Data

In [ ]:
df.isna().sum()

- There are **11** and **7** missing values in columns `V1` and `V2`
- We will treat them later

### Check for duplicate values in Train Data

In [ ]:
# let's check for duplicate values in the data
df.duplicated().sum()

There are no duplicate values in Train data

### Check for duplicate values in Test Data

In [ ]:
# let's check for duplicate values in test data
df_test.duplicated().sum()

There are no duplicate values in Test data

### Statistical summary of the Train Data

In [ ]:
df.describe().T

Observations:

- Most of the sensors appear to be normally distributed or close to normally distributed with the mean and median values for most sensors similar to each other
- There are 2 sensors: V1 and V2 with a few missing values

## EDA

### Plotting histograms and boxplots for all the variables

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

### Plotting all the features at one go

In [ ]:
for feature in df.columns:
    histogram_boxplot(df, feature, figsize=(12, 7), kde=False, bins=None)

Observations:

- Most of the predictor variables appear to have a normal to approximately normal distribution
- There appears to be a number of outliers on both the left and right for all predictor variables

### Function to plot labeled bar plots for the Target variable

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 2, 6))
    else:
        plt.figure(figsize=(n + 2, 6))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        palette="Paired",
        order=data[feature].value_counts().index[:n],
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

### **Target Variable**

In [ ]:
# Show class distribution in target variable in training data

df['Target'].value_counts()

In [ ]:
# Show class distribution in target variable in training data visually

labeled_barplot(df, "Target", perc=True)

In [ ]:
# Show class distribution in target variable in Test data

df_test['Target'].value_counts()

In [ ]:
# Show class distribution in target variable in Test data visually

labeled_barplot(df_test, "Target", perc=True)

- There appears to be class imbalance in the target variable with the **0- No Failure** class representing 94.5% of the data and the **1- Failure** class representing 5.5% of the data
- Proportion of the classes is same in both training and test data

## Data Pre-processing

#### Split Target variable from predictor variables

In [ ]:
# Separating target variable from predictor variables in training data
X=df.drop(['Target'],axis=1)
y=df['Target']

In [ ]:
# Separating target variable from predictor variables in test data
X_test=df_test.drop(['Target'],axis=1)
y_test=df_test['Target']

#### Split Train dataset into training and validation set

In [ ]:
#Splitting training dataset into train and validation sets (75:25)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)


In [ ]:
# Display training, validation and test data shapes
print(X_train.shape, X_val.shape, X_test.shape)

### Missing Value Treatment

#### Imputing missing values using median value

In [ ]:
# Create an instance of Simple-Imputer
imputer = SimpleImputer(strategy="median")

#### Fit and Transform imputer on datasets

In [ ]:
# Fit and transform the train data
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Transform the validation data
X_val =  pd.DataFrame(imputer.transform(X_val), columns=X_val.columns)

# Transform the test data
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

#### Verify all missing values have been treated

In [ ]:
# Checking that no column has missing values in train, validation or test sets
print("Training Data")
print(X_train.isna().sum())
print("-" * 200)

print("Validation Data")
print(X_val.isna().sum())
print("-" * 200)

print("Test Data")
print(X_test.isna().sum())
print("-" * 200)

## Model Building

### Model evaluation criterion

The nature of predictions made by the classification model will translate as follows:

- True positives (TP) are failures correctly predicted by the model.
- False negatives (FN) are real failures in a generator where there is no detection by model.
- False positives (FP) are failure detections in a generator where there is no failure.

### Which metric to optimize?

* We need to choose the metric which will ensure that the maximum number of generator failures are predicted correctly by the model.
* We would want Recall to be maximized as greater the Recall, the higher the chances of minimizing false negatives.
* We want to minimize false negatives because if a model predicts that a machine will have no failure when there will be a failure, it will increase the maintenance cost.

**Let's define a function to output different metrics (including recall) on the train and test set and a function to show confusion matrix so that we do not have to use the same code repetitively while evaluating models.**

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "Accuracy": acc,
            "Recall": recall,
            "Precision": precision,
            "F1": f1

        },
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

### Defining scorer to be used for cross-validation and hyperparameter tuning

- We want to reduce false negatives and will try to maximize "Recall".
- To maximize Recall, we can use Recall as a **scorer** in cross-validation and hyperparameter tuning.

In [ ]:
# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.recall_score)

### Model Building with original data

- Let's start by building different models using KFold and cross_val_score

- We will implement the following models:
  - Logistic Regression
  - Decision Tree
  - Bagging Classifier
  - Random Forest Classifier
  - Gradient Boosting Classifier
  - Ada Boost Classifier
  - XG Boost Classifier

- `Stratified K-Folds cross-validation` provides dataset indices to split data into train/validation sets. Split dataset into k consecutive folds  keeping the distribution of both classes in each fold the same as the target variable. Each fold is then used once as validation while the k - 1 remaining folds form the training set

- Once the models are built, we will evaluate the performance of the models using the validation dataset

#### Original data model building

In [ ]:
models = []  # Empty list to store all the models

# Appending models into the list
models.append(("Logistic regression", LogisticRegression(random_state=1)))
models.append(("dtree", DecisionTreeClassifier(random_state=1)))
models.append(("Bagging", BaggingClassifier(random_state=1)))
models.append(("Random forest", RandomForestClassifier(random_state=1)))
models.append(("GBM", GradientBoostingClassifier(random_state=1)))
models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))

results1 = []  # Empty list to store all model's CV scores
names = []  # Empty list to store name of the models


# loop through all models to get the mean cross validated score
print("\n" "Cross-Validation Cost:" "\n")

for name, model in models:
    kfold = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=1
    )  # Setting number of splits equal to 5
    cv_result = cross_val_score(
        estimator=model, X=X_train, y=y_train, scoring=scorer, cv=kfold
    )
    results1.append(cv_result)
    names.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\n" "Validation Performance:" "\n")

for name, model in models:
    model.fit(X_train, y_train)
    scores = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))


Cross-Validation Cost:

Logistic regression: 0.48292682926829267
dtree: 0.7335365853658538
Bagging: 0.7347560975609755


In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(10, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results1)
ax.set_xticklabels(names)

plt.show()

### Comments on Model building with original data

- Recall scores in the validation set range from 0.46 to 0.76 and is not very high

- Most of the models appear to be underfit with low CV score and low score on validation set

- Top 3 models with highest recall on validation set appears to be Random Forest, Decision Tree and XGBoost

- CV scores of most of the models are similar to the scores obtained on the validation set

- This suggests that most of the models seem to generalize reasonably well

- We will evaluate if model performances could be improved by over-sampling or undersampling techniques so that minority class could be predicted better

### Model Building with Oversampled data

- To avoid giving greater preference to the prediction of the majority class i.e **Class 0**, we are using over-sampling techniques to balance class representation
- Frequency of the Minority **Class 1** is increased to provide a more balanced class representation

In [ ]:
# Class size of Training Data Before Over-sampling
print("Before OverSampling, counts of label '1': {}".format(sum(y_train == 1)))
print("Before OverSampling, counts of label '0': {} \n".format(sum(y_train == 0)))

In [ ]:
# Synthetic Minority Over Sampling Technique
sm = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=1)
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)

In [ ]:
# Class size of Training Data after Over-sampling
print("After OverSampling, counts of label '1': {}".format(sum(y_train_over == 1)))
print("After OverSampling, counts of label '0': {} \n".format(sum(y_train_over == 0)))

In [ ]:
# Shape of Training Data after Over-sampling
print("After OverSampling, the shape of train_X: {}".format(X_train_over.shape))
print("After OverSampling, the shape of train_y: {} \n".format(y_train_over.shape))

Comments on Over-sampling Training data:

- Minority **Class 1** has only 1640 entries before over-sampling
- After oversampling, **Class 1** has 28360 entries, thus making a 50:50 class representations between Class 1 and Class 0
- Lets evaluate if this improves the performance of the ML models

#### Oversampled data Model Building

In [ ]:
models_over = []  # Empty list to store all the models

# Appending models into the list
models_over.append(("Logistic regression", LogisticRegression(random_state=1)))
models_over.append(("dtree", DecisionTreeClassifier(random_state=1)))
models_over.append(("Bagging", BaggingClassifier(random_state=1)))
models_over.append(("Random forest", RandomForestClassifier(random_state=1)))
models_over.append(("GBM", GradientBoostingClassifier(random_state=1)))
models_over.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models_over.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))

results2 = []  # Empty list to store all model's CV scores
names_over = []  # Empty list to store name of the models


# loop through all models to get the mean cross validated score
print("\n" "Cross-Validation Cost:" "\n")

for name, model in models_over:
    kfold = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=1
    )  # Setting number of splits equal to 5
    cv_result = cross_val_score(
        estimator=model, X=X_train_over, y=y_train_over, scoring=scorer, cv=kfold
    )
    results2.append(cv_result)
    names_over.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\n" "Validation Performance:" "\n")

for name, model in models_over:
    model.fit(X_train_over, y_train_over)
    scores = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(10, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results2)
ax.set_xticklabels(names_over)

plt.show()

### Comments on Model building with oversampled data

- There is a significant improvement in the CV scores of all the models compared to the models built using original data

- CV recall scores in the models range from 0.87 to 0.98

- Broadly, the performance of the models seem to have improved using oversampled data

- However, the CV scores of most models seem to be systematically higher than the scores obtained on the validation set suggesting a moderate degree of overfitting

- This suggests that most of the models seems to generalize to some extent but not exceptionally well

- Top 3 models with highest recall on validation set appears to be Gradient Boosting, XGBoost and Random Forest

- Hyperparameter tuning may be helpful to further improve the performances of these models using oversampled data as well as help reduce the amount of overfitting

### Model Building with Undersampled data

- To avoid giving greater preference to the prediction of the majority class i.e **Class 0**, we are using under-sampling techniques to balance class representation
- Frequency of the Majority **Class 0** is decreased to achieve more balanced class representation in undersampling

In [ ]:
# Class size of Training Data Before Undersampling
print("Before UnderSampling, counts of label '1': {}".format(sum(y_train == 1)))
print("Before UnderSampling, counts of label '0': {} \n".format(sum(y_train == 0)))

In [ ]:
# Random undersampler for under sampling the data
rus = RandomUnderSampler(random_state=1, sampling_strategy=1)
X_train_un, y_train_un = rus.fit_resample(X_train, y_train)

In [ ]:
# Class size of Training Data after Undersampling
print("After UnderSampling, counts of label '1': {}".format(sum(y_train_un == 1)))
print("After UnderSampling, counts of label '0': {} \n".format(sum(y_train_un == 0)))

In [ ]:
# Shape of Training Data after Undersampling
print("After UnderSampling, the shape of train_X: {}".format(X_train_un.shape))
print("After UnderSampling, the shape of train_y: {} \n".format(y_train_un.shape))

#### Undersampled data model building

In [ ]:
models_un = []  # Empty list to store all the models

# Appending models into the list
models_un.append(("Logistic regression", LogisticRegression(random_state=1)))
models_un.append(("dtree", DecisionTreeClassifier(random_state=1)))
models_un.append(("Bagging", BaggingClassifier(random_state=1)))
models_un.append(("Random forest", RandomForestClassifier(random_state=1)))
models_un.append(("GBM", GradientBoostingClassifier(random_state=1)))
models_un.append(("Adaboost", AdaBoostClassifier(random_state=1)))
models_un.append(("Xgboost", XGBClassifier(random_state=1, eval_metric="logloss")))

results3 = []  # Empty list to store all model's CV scores
names_un = []  # Empty list to store name of the models


# loop through all models to get the mean cross validated score
print("\n" "Cross-Validation Cost:" "\n")

for name, model in models_un:
    kfold = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=1
    )  # Setting number of splits equal to 5
    cv_result = cross_val_score(
        estimator=model, X=X_train_un, y=y_train_un, scoring=scorer, cv=kfold
    )
    results3.append(cv_result)
    names_un.append(name)
    print("{}: {}".format(name, cv_result.mean()))

print("\n" "Validation Performance:" "\n")

for name, model in models_un:
    model.fit(X_train_un, y_train_un)
    scores = recall_score(y_val, model.predict(X_val))
    print("{}: {}".format(name, scores))

In [ ]:
# Plotting boxplots for CV scores of all models defined above
fig = plt.figure(figsize=(10, 7))

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)

plt.boxplot(results3)
ax.set_xticklabels(names_un)

plt.show()

### Comments on Model building with undersampled data

- There is significant improvement in the CV scores of all the models compared to the models built using original data

- CV recall scores in the models range from 0.85 to 0.9

- The CV scores are generally slightly lower than the equivalent models built using oversampling techniques and thus perform slightly worse compared to models using oversampling

- However, the CV scores of most models seem to be very similar to the scores obtained on the validation set

- This suggests that the models using undersampled data seem to generalize quite well, slightly better than the models built using oversampling

- Top 3 models with highest recall on validation set appears to be Gradient Boosting, Random Forest and XGBoost

- Hyperparameter tuning may be helpful to further improve the performances of these models using undersampled data

## Hyperparameter Tuning

- Hyperparameter tuning can help improve model performance as well help to reduce both overfitting or underfitting

- Generally the **default models using the original data** with imbalanced classes performs poorly (low recall score) and appears to be underfit (poor performance in both training and validation)

- We can attempt to improve performance using hyperparameter tuning in these models

- Models using **oversampling techniques** have the best performance but appear to be slightly overfit with validation data scores systematically lower than the CV scores

- We can use hyperparameter tuning to see if we could reduce the overfitting and improve performance

- Models using **undersampling techniques** have slightly lower performance scores compared to the models using oversampled data but appear to generalize better with very similar CV scores and validation scores.

- We can use hyperparameter tuning to see if we could primarily improve performance

### Approach for Hyperparameter tuning

- In terms of performance based on CV score alone, few of the top five models are
  - Random Forest with oversampled data,
  - Bagging classifier with oversampled data,
  - Decision Tree with oversampled data,
  - Gradient Boosting with oversampled data
  - XGB with oversampled data

- However, all of these models suffer from slight overfitting and we will tune hyperparameters to see how much we can improve them and reduce overfitting

- In terms of both performance and model generalization, few of the top five models are
 - Random Forest with undersampled data,
 - Gradient Boosting with undersampled data,
 - XGB with undersampled data,
 - Bagging classifier with undersampled data
 - Ada Boost with undersampled data

- We will focus on improving performance for these models using hyperparameter tuning

- Instead of selecting just a few models for hyperparameter tuning , we will be tuning 15 models (all models except Bagging Classifier and Ada Boost Classifier) with default, over- and undersampled data with the objective of either
 - improve performance and reduce underfitting (Default models)
 - reduce overfitting and generalize better as well as further improve performance (Models using oversampled data)
 - improve performance (Models using undersampled data)

- **Bagging Classifier models have been left out** from tuning because they are extremely time intensive for computation and produce very poorly generalized, highly overfit models **[NOTE: These models were tuned earlier but excluded in final solution]**

- **Ada Boost Classifier models have been left out** from tuning because they are are extremely time intensive for computation and their pre-tuned performance isn't significantly better compared to the other models **[NOTE: These models were tuned earlier but excluded in final solution]**

### Selection of Top Models for Hyperparamater Tuning

- Although we will be tuning additional models to see how hyper-parameter tuning affects performance and generalization across different models, we will highlight here top 7 models to be selected in case there are time constraints to tune all models

- Top models to select for Hyperparameter tuning are as follows
  - (1) Random Forest with Oversampling
  - (2) Bagging Classifier with Oversampling
  - (3) Gradient Boosting with Oversampling
  - (4) XGBoost with Oversampling
  - (5) Random Forest with Undersampling
  - (6) Gradient Boosting with Undersampling
  - (7) XGBoost with Undersampling

- Reasons for selection of the above models in scenarios where time is limited and all 21 models cannot be tuned are
  - Models (1) to (4) are selected because they perform really well with high recall CV score but suffer from slight overfitting. Hyperparamater tuning may help to generalize them without losing much performance
  - Models (5) to (7) are selected because they generalize quite well with relatively high performance. Tuning may help further improve performance without losing generalization
  - Model (2) should be skipped if there are time limitations as it is the most computationally expensive amongst these 7 models

- NOTE: This is done here for consistency purposes related to the grading rubric

### Sample Parameter Grids

**Hyperparameter tuning can take a long time to run, so to avoid that time complexity - you can use the following grids, wherever required.**

- For Gradient Boosting:

param_grid = {
    "n_estimators": np.arange(100,150,25),
    "learning_rate": [0.2, 0.05, 1],
    "subsample":[0.5,0.7],
    "max_features":[0.5,0.7]
}

- For Adaboost:

param_grid = {
    "n_estimators": [100, 150, 200],
    "learning_rate": [0.2, 0.05],
    "base_estimator": [DecisionTreeClassifier(max_depth=1, random_state=1), DecisionTreeClassifier(max_depth=2, random_state=1), DecisionTreeClassifier(max_depth=3, random_state=1),
    ]
}

- For Bagging Classifier:

param_grid = {
    'max_samples': [0.8,0.9,1],
    'max_features': [0.7,0.8,0.9],
    'n_estimators' : [30,50,70],
}

- For Random Forest:

param_grid = {
    "n_estimators": [200,250,300],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}

- For Decision Trees:

param_grid = {
    'max_depth': np.arange(2,6),
    'min_samples_leaf': [1, 4, 7],
    'max_leaf_nodes' : [10, 15],
    'min_impurity_decrease': [0.0001,0.001]
}

- For Logistic Regression:

param_grid = {'C': np.arange(0.1,1.1,0.1)}

- For XGBoost:

param_grid={
    'n_estimators': [150, 200, 250],
    'scale_pos_weight': [5,10],
    'learning_rate': [0.1,0.2],
    'gamma': [0,3,5],
    'subsample': [0.8,0.9]
}

### 1. Hyperparameter tuning Logistic Regression with original data

In [ ]:
%%time
# defining model
Model = LogisticRegression(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {'C': np.arange(0.1,1.1,0.1)}


#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_lr1 = LogisticRegression(random_state=1,C=0.1)

tuned_lr1.fit(X_train,y_train)

In [ ]:
# Calculating different metrics on train set
tuned_lr1_train_perf = model_performance_classification_sklearn(tuned_lr1, X_train, y_train)
tuned_lr1_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_lr1, X_train, y_train)

In [ ]:
# Calculating different metrics on validation set
tuned_lr1_val_perf = model_performance_classification_sklearn(tuned_lr1, X_val, y_val)
tuned_lr1_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_lr1, X_val, y_val)

#### Comments on hyperparameter tuning Logistic regression model with original data

- Training and validation recall scores are very poor (< 0.5)

- Recall scores is very similar to non-tuned logistic regression models with original data and havent improved

- Model is extremely poor in correctly predicting the minority class (class '1')

### 2. Hyperparameter tuning Logistic Regression with oversampled data

In [ ]:
%%time
# defining model
Model = LogisticRegression(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {'C': np.arange(0.1,1.1,0.1)}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_over,y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_lr2 = LogisticRegression(random_state=1,C=0.1)

tuned_lr2.fit(X_train_over,y_train_over)

In [ ]:
# Calculating different metrics on train set
tuned_lr2_train_perf = model_performance_classification_sklearn(tuned_lr2, X_train_over, y_train_over)
tuned_lr2_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_lr2, X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on validation set
tuned_lr2_val_perf = model_performance_classification_sklearn(tuned_lr2, X_val, y_val)
tuned_lr2_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_lr2, X_val, y_val)

#### Comments on hyperparameter tuning Logistic regression model with oversampled data

- Recall has significantly improved to 0.84 in validation set compared to the model using original data

- Training and validation set recall scores are similar suggesting reasonable model generalization with respect to recall

- Precision and F1 score are quite poor (0.28 and 0.42) in validation set and much lower compared to the scores in training set

- This suggests that the model performance and generalization are both very poor with respect to precision and F1 score

### 3. Hyperparameter tuning Logistic Regression with undersampled data

In [ ]:
%%time
# defining model
Model = LogisticRegression(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {'C': np.arange(0.1,1.1,0.1)}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_un,y_train_un)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_lr3 = LogisticRegression(random_state=1,C=0.1)

tuned_lr3.fit(X_train_un,y_train_un)

In [ ]:
# Calculating different metrics on train set
tuned_lr3_train_perf = model_performance_classification_sklearn(tuned_lr3, X_train_un, y_train_un)
tuned_lr3_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_lr3, X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on validation set
tuned_lr3_val_perf = model_performance_classification_sklearn(tuned_lr3, X_val, y_val)
tuned_lr3_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_lr3, X_val, y_val)

#### Comments on hyperparameter tuning Logistic regression model with undersampled data

- Recall in validation set is very similar to model using oversampling

- Model seems to generalize well with respect to recall score

- Precision and F1 score are very poor (0.26 and 0.405) in validation set and much lower compared to the scores in training set

- This suggests that the model performance and generalization are both very poor with respect to precision and F1 score

### 4. Hyperparameter tuning Decision tree with original data

In [ ]:
%%time
# defining model
Model = DecisionTreeClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {'max_depth': np.arange(2,6),
              'min_samples_leaf': [1, 4, 7],
              'max_leaf_nodes' : [10,15],
              'min_impurity_decrease': [0.0001,0.001] }

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_dt1 = DecisionTreeClassifier(random_state=1,min_samples_leaf=7, min_impurity_decrease = 0.0001, max_leaf_nodes = 15, max_depth = 5)

tuned_dt1.fit(X_train,y_train)

In [ ]:
# Calculating different metrics on train set
tuned_dt1_train_perf = model_performance_classification_sklearn(tuned_dt1, X_train, y_train)
tuned_dt1_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_dt1, X_train, y_train)

In [ ]:
# Calculating different metrics on validation set
tuned_dt1_val_perf = model_performance_classification_sklearn(tuned_dt1, X_val, y_val)
tuned_dt1_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_dt1, X_val, y_val)

#### Comments on hyperparameter tuning Decision Tree model with original data

- Recall score is not very high (~0.65)

- Recall scores is poorer compared to pre-tuned models and this may partly arise from the range of parameters in the parameter grid and the use of Randomized grid search as opposed to the more extensive grid search

- Model generalizes well with respect to most performance metrics and is not very overfit

### 5. Hyperparameter tuning Decision tree with oversampled data

In [ ]:
%%time
# defining model
Model = DecisionTreeClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {'max_depth': np.arange(2,6),
              'min_samples_leaf': [1, 4, 7],
              'max_leaf_nodes' : [10,15],
              'min_impurity_decrease': [0.0001,0.001] }

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_over,y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_dt2 = DecisionTreeClassifier(random_state=1,min_samples_leaf=7, min_impurity_decrease = 0.0001, max_leaf_nodes = 15, max_depth = 5)

tuned_dt2.fit(X_train_over,y_train_over)

In [ ]:
# Calculating different metrics on train set
tuned_dt2_train_perf = model_performance_classification_sklearn(tuned_dt2, X_train_over, y_train_over)
tuned_dt2_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_dt2, X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on validation set
tuned_dt2_val_perf = model_performance_classification_sklearn(tuned_dt2, X_val, y_val)
tuned_dt2_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_dt2, X_val, y_val)

#### Comments on hyperparameter tuning Decision Tree model with oversampled data

- Recall score has improved to 0.81 in validation set compared to model built with original data

- Model appears slightly overfit with respect to recall with validation score slightly lower compared to training set score

- Model appears quite overfit with respect to precision and F1 score with model performing poorly on validation data


### 6. Hyperparameter tuning Decision tree with undersampled data

In [ ]:
%%time
# defining model
Model = DecisionTreeClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {'max_depth': np.arange(2,20),
              'min_samples_leaf': [1, 2, 5, 7],
              'max_leaf_nodes' : [5, 10,15],
              'min_impurity_decrease': [0.0001,0.001] }

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_un,y_train_un)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_dt3 = DecisionTreeClassifier(random_state=1,min_samples_leaf=1, min_impurity_decrease = 0.001, max_leaf_nodes = 15, max_depth = 11)

tuned_dt3.fit(X_train_un,y_train_un)

In [ ]:
# Calculating different metrics on train set
tuned_dt3_train_perf = model_performance_classification_sklearn(tuned_dt3, X_train_un, y_train_un)
tuned_dt3_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_dt3, X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on validation set
tuned_dt3_val_perf = model_performance_classification_sklearn(tuned_dt3, X_val, y_val)
tuned_dt3_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_dt3, X_val, y_val)

#### Comments on hyperparameter tuning Decision Tree model with undersampled data

- Recall score in validation set is similar to oversampled data model

- Model appears slightly overfit with respect to recall with validation score slightly lower compared to training score

- Model is very overfit with respect to precision and F1 score with model performing poorly on validation data

- Limited range of hyperparameters and use of Randomized Grid search may be responsible for similar poor performances in models using oversampled and undersampled techniques

### 7. Hyperparameter tuning Random Forest Classifer with original data

In [ ]:
%%time
# defining model
Model = RandomForestClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": [200,250,300],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_rf1 = RandomForestClassifier(random_state=1,n_estimators = 300, min_samples_leaf = 1, max_samples = 0.6, max_features = 'sqrt')

tuned_rf1.fit(X_train,y_train)

In [ ]:
# Calculating different metrics on train set
tuned_rf1_train_perf = model_performance_classification_sklearn(tuned_rf1, X_train, y_train)
tuned_rf1_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_rf1, X_train, y_train)

In [ ]:
# Calculating different metrics on validation set
tuned_rf1_val_perf = model_performance_classification_sklearn(tuned_rf1, X_val, y_val)
tuned_rf1_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_rf1, X_val, y_val)

#### Comments on hyperparameter tuning Random Forest model with original data

- Model appears to be overfitting with respect to recall score

- Model validation recall is very similar to pre-tuned model and not much improved

- Model precision and accuracy is high and generalizes well but the model does a poorer job in predicting the minority class

### 8. Hyperparameter tuning Random Forest Classifer with oversampled data

In [ ]:
%%time
# defining model
Model = RandomForestClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": [200,250,300],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_over,y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_rf2 = RandomForestClassifier(random_state=1,n_estimators = 250, min_samples_leaf = 1, max_samples = 0.6, max_features = 'sqrt')

tuned_rf2.fit(X_train_over,y_train_over)

In [ ]:
# Calculating different metrics on train set
tuned_rf2_train_perf = model_performance_classification_sklearn(tuned_rf2, X_train_over, y_train_over)
tuned_rf2_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_rf2, X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on validation set
tuned_rf2_val_perf = model_performance_classification_sklearn(tuned_rf2, X_val, y_val)
tuned_rf2_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_rf2, X_val, y_val)

#### Comments on hyperparameter tuning Random Forest Classifier model with Oversampled data

- Model performs very well on the validation set with very high performance metrics

- However the model appears to be slightly overfit with near perfect training scores but slightly lower scores on the validation set (Training recall is ~1; Validation recall is 0.87)

- Model generalizes reasonably well but not great

- Model performance is similar to pre-tuned random forest model with oversampled data

### 9. Hyperparameter tuning Random Forest Classifer with undersampled data

In [ ]:
%%time
# defining model
Model = RandomForestClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": [200,250,300],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_un,y_train_un)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_rf3 = RandomForestClassifier(random_state=1, n_estimators = 200, min_samples_leaf = 2, max_samples = 0.4, max_features = 'sqrt')

tuned_rf3.fit(X_train_un,y_train_un)

In [ ]:
# Calculating different metrics on train set
tuned_rf3_train_perf = model_performance_classification_sklearn(tuned_rf3, X_train_un, y_train_un)
tuned_rf3_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_rf3, X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on validation set
tuned_rf3_val_perf = model_performance_classification_sklearn(tuned_rf3, X_val, y_val)
tuned_rf3_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_rf3, X_val, y_val)

#### Comments on hyperparameter tuning Random Forest Classifier model with Undersampled data

- Model performs very well on the validation set with respect to accuracy and recall and appears to generalize with respect to these metrics as well

- Model appears to be quite overfit with respect to precision and F1 score and does not generalize

### 10. Hyperparameter tuning Gradient Boosting Classifer with original data

In [ ]:
%%time
# defining model
Model = GradientBoostingClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(100,150,25),
    "learning_rate": [0.2, 0.05, 1],
    "subsample":[0.5,0.7],
    "max_features":[0.5,0.7]
}
#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_gb1 = GradientBoostingClassifier(random_state=1, subsample = 0.7, n_estimators = 125, max_features = 0.5, learning_rate = 0.2)

tuned_gb1.fit(X_train,y_train)

In [ ]:
# Calculating different metrics on train set
tuned_gb1_train_perf = model_performance_classification_sklearn(tuned_gb1, X_train, y_train)
tuned_gb1_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_gb1, X_train, y_train)

In [ ]:
# Calculating different metrics on validation set
tuned_gb1_val_perf = model_performance_classification_sklearn(tuned_gb1, X_val, y_val)
tuned_gb1_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_gb1, X_val, y_val)

#### Comments on hyperparameter tuning Gradient Boosting Classifier model with Original data

- Model has a decent but not great recall score on validation set (0.75)

- Model appears to be slightly overfit with respect to recall and F1 score and generalizes reasonably well but not great.

- Model has very high accuracy and precision on validation set and generalizes well with respect to these metrics

### 11. Hyperparameter tuning Gradient Boosting Classifer with oversampled data

In [ ]:
%%time
# defining model
Model = GradientBoostingClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(100,150,25),
    "learning_rate": [0.2, 0.05, 1],
    "subsample":[0.5,0.7],
    "max_features":[0.5,0.7]
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_over,y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_gb2 = GradientBoostingClassifier(random_state=1, subsample = 0.7, n_estimators = 125, max_features = 0.5, learning_rate = 1)

tuned_gb2.fit(X_train_over,y_train_over)

In [ ]:
# Calculating different metrics on train set
tuned_gb2_train_perf = model_performance_classification_sklearn(tuned_gb2, X_train_over, y_train_over)
tuned_gb2_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_gb2, X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on validation set
tuned_gb2_val_perf = model_performance_classification_sklearn(tuned_gb2, X_val, y_val)
tuned_gb2_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_gb2, X_val, y_val)

#### Comments on hyperparameter tuning Gradient Boosting Classifier model with Oversampled data

- Model has high accuracy and recall but pays the price in precision and F1 score on the validation set

- Model appears overfitting particularly in respect to precision and F1 score but should generalize reasonably well with respect to recall and accuracy

### 12. Hyperparameter tuning Gradient Boosting Classifer with undersampled data

In [ ]:
%%time
# defining model
Model = GradientBoostingClassifier(random_state=1)

# Parameter grid to pass in RandomSearchCV
param_grid = {
    "n_estimators": np.arange(100,150,25),
    "learning_rate": [0.2, 0.05, 1],
    "subsample":[0.5,0.7],
    "max_features":[0.5,0.7]
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_un,y_train_un)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_gb3 = GradientBoostingClassifier(random_state=1, subsample = 0.5, n_estimators = 125, max_features = 0.5, learning_rate = 0.2)

tuned_gb3.fit(X_train_un,y_train_un)

In [ ]:
# Calculating different metrics on train set
tuned_gb3_train_perf = model_performance_classification_sklearn(tuned_gb3, X_train_un, y_train_un)
tuned_gb3_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_gb3, X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on validation set
tuned_gb3_val_perf = model_performance_classification_sklearn(tuned_gb3, X_val, y_val)
tuned_gb3_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_gb3, X_val, y_val)

#### Comments on hyperparameter tuning Gradient Boosting Classifier model with Undersampled data

- Model has high accuracy and recall but pays the price in precision and F1 score on the validation set

- Model appears very overfit in respect to precision and F1 score but should generalize reasonably well with respect to recall and accuracy

### 13. Hyperparameter tuning XGBoost Classifer with original data

In [ ]:
%%time
# defining model
Model = XGBClassifier(random_state=1, eval_metric="logloss")

# Parameter grid to pass in RandomSearchCV
param_grid={
    'n_estimators': [150, 200, 250],
    'scale_pos_weight': [5,10],
    'learning_rate': [0.1,0.2],
    'gamma': [0,3,5],
    'subsample': [0.8,0.9]
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_xgb1 = XGBClassifier(random_state=1, subsample = 0.9, scale_pos_weight = 10, n_estimators = 150, learning_rate = 0.2, gamma = 3)

tuned_xgb1.fit(X_train,y_train)

In [ ]:
# Calculating different metrics on train set
tuned_xgb1_train_perf = model_performance_classification_sklearn(tuned_xgb1, X_train, y_train)
tuned_xgb1_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_xgb1, X_train, y_train)

In [ ]:
# Calculating different metrics on validation set
tuned_xgb1_val_perf = model_performance_classification_sklearn(tuned_xgb1, X_val, y_val)
tuned_xgb1_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_xgb1, X_val, y_val)

#### Comments on hyperparameter tuning XGBoost Classifier model with Original data

- Model performs really well with most performance metrics with validation recall score of 0.88

- Model also generalizes quite well with most metrics particulary accuracy and precision

- Model recall is significantly improved from the non-tuned XG Boost model using original data


### 14. Hyperparameter tuning XGBoost Classifer with oversampled data

In [ ]:
%%time
# defining model
Model = XGBClassifier(random_state=1, eval_metric="logloss")

# Parameter grid to pass in RandomSearchCV
param_grid={
    'n_estimators': [150, 200, 250],
    'scale_pos_weight': [5,10],
    'learning_rate': [0.1,0.2],
    'gamma': [0,3,5],
    'subsample': [0.8,0.9]
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_over,y_train_over)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_xgb2 = XGBClassifier(random_state=1, subsample = 0.9, scale_pos_weight = 10, n_estimators = 200, learning_rate = 0.2, gamma = 0)

tuned_xgb2.fit(X_train_over,y_train_over)

In [ ]:
# Calculating different metrics on train set
tuned_xgb2_train_perf = model_performance_classification_sklearn(tuned_xgb2, X_train_over, y_train_over)
tuned_xgb2_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_xgb2, X_train_over, y_train_over)

In [ ]:
# Calculating different metrics on validation set
tuned_xgb2_val_perf = model_performance_classification_sklearn(tuned_xgb2, X_val, y_val)
tuned_xgb2_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_xgb2, X_val, y_val)

#### Comments on hyperparameter tuning XGBoost Classifier model with Oversampled data

- Model performs really well with respect to recall in validation set (0.92)

- Model shows decent generalization with respect to accuracy and recall

- Model appears to be very overfit with respect to precision and F1 score

- Model recall score on the validation set is improved from the non-tuned XGB model with oversampled data

### 15. Hyperparameter tuning XGBoost Classifer with undersampled data

In [ ]:
%%time
# defining model
Model = XGBClassifier(random_state=1, eval_metric="logloss")

# Parameter grid to pass in RandomSearchCV
param_grid={
    'n_estimators': [150, 200, 250],
    'scale_pos_weight': [5,10],
    'learning_rate': [0.1,0.2],
    'gamma': [0,3,5],
    'subsample': [0.8,0.9]
}

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(estimator=Model, param_distributions=param_grid, n_iter=10, n_jobs = -1, scoring=scorer, cv=5, random_state=1)

#Fitting parameters in RandomizedSearchCV
randomized_cv.fit(X_train_un,y_train_un)

print("Best parameters are {} with CV score={}:" .format(randomized_cv.best_params_,randomized_cv.best_score_))

In [ ]:
# Creating new pipeline with best parameters
tuned_xgb3 = XGBClassifier(random_state=1, subsample = 0.8, scale_pos_weight = 10, n_estimators = 150, learning_rate = 0.1, gamma = 3)

tuned_xgb3.fit(X_train_un,y_train_un)

In [ ]:
# Calculating different metrics on train set
tuned_xgb3_train_perf = model_performance_classification_sklearn(tuned_xgb3, X_train_un, y_train_un)
tuned_xgb3_train_perf

In [ ]:
#Confusion matrix on train set
confusion_matrix_sklearn(tuned_xgb3, X_train_un, y_train_un)

In [ ]:
# Calculating different metrics on validation set
tuned_xgb3_val_perf = model_performance_classification_sklearn(tuned_xgb3, X_val, y_val)
tuned_xgb3_val_perf

In [ ]:
#Confusion matrix on validation set
confusion_matrix_sklearn(tuned_xgb3, X_val, y_val)

#### Comments on hyperparameter tuning XGBoost Classifier model with Undersampled data

- Model performs really well with respect to recall in validation set (0.94)

- Model shows decent generalization with respect to recall score in validation set

- Model appears to be extremely overfit with respect to precision and F1 score and overfit with respect to accuracy

- Model recall score on the validation set is improved from the non-tuned XGB model with oversampled data


## Model performance comparison and choosing the final model

#### Training Performance comparison

In [ ]:
# Model Name as Index
index = pd.Index(
    [
        "Log_Reg_Tuned",
        "Log_Reg_Tuned_Over",
        "Log_Reg_Tuned_Under",
        "Decision_Tree_Tuned",
        "Decision_Tree_Tuned_Over",
        "Decision_Tree_Tuned_Under",
        "Random_Forest_Tuned",
        "Random_Forest_Tuned_Over",
        "Random_Forest_Tuned_Under",
        "Gradient_Boost_Tuned",
        "Gradient_Boost_Tuned_Over",
        "Gradient_Boost_Tuned_Under",
        "XGBoost_Tuned",
        "XGBoost_Tuned_Over",
        "XGBoost_Tuned_Under",

    ]
)

# Create DataFrame with different models as rows, and performance metrics as columns
model_train_perf_comp = pd.concat(
    [
        tuned_lr1_train_perf,
        tuned_lr2_train_perf,
        tuned_lr3_train_perf,
        tuned_dt1_train_perf,
        tuned_dt2_train_perf,
        tuned_dt3_train_perf,
        tuned_rf1_train_perf,
        tuned_rf2_train_perf,
        tuned_rf3_train_perf,
        tuned_gb1_train_perf,
        tuned_gb2_train_perf,
        tuned_gb3_train_perf,
        tuned_xgb1_train_perf,
        tuned_xgb2_train_perf,
        tuned_xgb3_train_perf,

    ],
    ignore_index=True,
)

# set model names as index
model_train_perf_comp = model_train_perf_comp.set_index(index)

print('*'*50)
print("Training Performance Comparison between models")
print('*'*50)
print("\n ")
print("-"*100)
model_train_perf_comp

## Summary of Performances of all tuned Models

- Most of the models **performs really well** in terms of **accuracy metric** (~ > 0.9 in most models)

- Most of the models also **generalize** quite well in terms of **data accuracy**

- In terms of recall, most of the models **using oversampled and undersampled data** outperforms the models using **original data**

- This isn't suprising given that these sampling techniques help improve the predictive power of the model on the minority class - **Class 1**

- Most of the models also **generalize relatively well** in terms of **recall metric**

- Most of the models using original data seem to generalize much better in terms of **precision and F1 score** compared to models using oversampled and undersampled data

- Most of the models using oversampled and undersampled data seem to be overfit in terms of **precision and F1 score**

- This is likely partly due to the emphasis on using recall as a scorer during cross validation which forces the models to correctly predict the minority class sometimes at the cost of incorrectly predicting the majority class when using over- and undersampling

- All of the 3 XGBoost Classifier model performs the best in terms of recall in the validation set data but the oversampling and undersampling models do not generalize very well in terms of F1 score and precision

- Top 2 models from all the tuned models that perform reasonably well on most metrics as well as generalizes are
  - Tuned Random Forest Classifier using Oversampled data
  - Tuned XGBoost Classifier using original data

## Final Model Selection

- In terms of selecting the final model to be sent for production, emphasis was given in choosing a model that had the following attributes
  - Model performs quite well in terms of recall as the largest priority is to predict the minority class accurately
  - Model generalizes quite well and do not suffer from overfitting in terms of recall so that model in production will perform well with unseen data
  - Model generalizes reasonably well in terms of other metrics such as precision and F1 score so that the model does not perform poorly with unseen data on those metrics and have fewer incorrect predictions

- Top 2 models from all the tuned models that meet the aforementioned criteria were the following
  - (1) Tuned Random Forest Classifier using Oversampled data
  - (2) Tuned XGBoost Classifier using original data

- XGBoost Classifier using Original Data is chosen as the final model because of the following reasons:
  - It has slightly higher recall score compared to the other model
  - It generalizes slightly better in terms of recall with the other top model
  - It generalizes better in terms of accuracy, precision and F1 score relative to the other top model
  - It is computationally less time intensive compared to the other top model
    - Wall time of only 10 min 42 s
    - Random Forest with Oversampled data has wall time of 27 mins and 38 s


## Test set performance on Final Model

In [ ]:
# Calculating different metrics on the test set
model_test = model_performance_classification_sklearn(tuned_xgb1, X_test, y_test)
print("Test performance:")
model_test

In [ ]:
#Confusion matrix on the test set
confusion_matrix_sklearn(tuned_xgb1, X_test, y_test)

Observations:

- Final model has a accuracy of **0.99** on test data

- Final model has a recall score of **0.87** on test data

- Final model has both the precision and F1 score of **0.86** on test data

- Final model performs quite well with unseen test data

- Final model generalizes quite well with unseen test data

## Feature Importance of Final Model

In [ ]:
feature_names = X.columns
importances = tuned_xgb1.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

Observations:

- Final model indicates that sensor V18 is one of the most critical sensors in accurately predicting machine failure
- This sensor has a relative importance of nearly 20% almost 3X more than the next most important sensor V36 in predicting machine failure
- Next top 3 sensors include V36 , V26 and V39 in decreasing order of importance

## Pipelines to build the final model

In [ ]:
# Build a pipeline with simple imputer to impute missing values and use the final model for production
Model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "XGB_Tuned",
            XGBClassifier(random_state=1,
                          subsample = 0.9,
                          scale_pos_weight = 10,
                          n_estimators = 150,
                          learning_rate = 0.2,
                          gamma = 3
            ),
        ),
    ]
)

In [ ]:
# Separating target variable and other variables
X1 = df.drop(columns="Target")
Y1 = df["Target"]

# Since we already have a separate test set, we don't need to divide data into train and test

X_test1 = df_test.drop(["Target"], axis=1)
y_test1 = df_test["Target"]



In [ ]:
# Treating missing values in train set
imputer = SimpleImputer(strategy="median")
X1 = imputer.fit_transform(X1)

# Treating missing values in the test set
X2 = imputer.transform(X_test1)

In [ ]:
# Fit the productionized model
Model.fit(X1, Y1)

In [ ]:
# Make predictions on the test data using the productionized model
Model.predict(X2)

# Conclusions

- A number of classification type machine learning models were utilized in order to accurately predict generator failure of wind turbines using sensor data

- The key objective was to identify generator failure or breaking before it happened in order to reduce maintenance cost given that the replacement cost (for predicting no failure when it actually fails) is 2X the repair cost+inspection cost (for correctly predicting failure)

- 7 different type of classifier models were utilized namely the following:
  - Logistic Regression
  - Decision Tree
  - Bagging Classifier
  - Random Forest Classifier
  - Gradient Boosting Classifier
  - Ada Boosting Classifier
  - XGBoost Classifier

- Dataset is highly imbalanced with the minority class predicting generator failure making up only 5% of the observations

- Oversampling and undersampling techniques were employed on the original data to improve the predictive power of the model on the minority class - **Class 1 denoting Failure**

- Hyperparameter tuning was performed on some of these models with the objective of improving performance as well as reducing overfitting and underfitting when applicable

- Two top models were identified based on performance particularly recall score as well as model generalization on unseen data. These are
  - (1) Tuned Random Forest Classifier using Oversampled data
  - (2) Tuned XGBoost Classifier using original data

- Of these 2 models, XGBoost Classifier using original data was chosen as the final model

- Recall score on the final model was 87% on the unseen test data.

- Model accuracy for the final model was 99% and precision and F1 score were 86% each

- Final model indictated sensor V18 to be the most important sensor in terms of predicting generator failure. Other 3 important sensors in decreasing order of importance are V36, V26 and V39

# Business Insights

- **Sensor V18** (followed by sensors V36, V26 and V39 to a lesser extent) are critical in terms of indicating pending generator failure

  - Renewind is recommended to pay very close attention to data coming from sensor V18 (and maybe potentially sensors V36, V26 and V39 as well). These sensors are somewhat more predictive of a generator failure relative to other sensors
  - Any indication of a looming generator failure from the data should be used to send out a technician for inspection and/or repair (if needed).

- ML models were built using **recall score** as the key metric for predicting generator failure. Instead, an alternative approach could be used where the **total service cost** is used as the metric with the objective being to lower this cost

  - Total service cost based on the different ML models could be computed using the confusion matrix
  - Total service cost could be computed by adding the following:
    - Cost of accurately predicting generator failure **[True Positive (TP)]** (Inspection cost + Repair cost)
    - Cost of accurately predicting no generator failure **[True Negative (TN)]** (No cost)
    - Cost of incorrectly predicting generator failure **[False Positive (FP)]** (Inspection cost only and no repair cost)
    - Cost of incorrectly predicting no generator failure when actual failure is happening **[False Negative (FN)]** ( Replacement cost)
  - Models with the lowest total service cost could be chosen as opposed to the model with best recall
  - This approach may help ReneWind incur lower costs in the maintenance of the generator in the long-run


- Renewind could be consulted on how critical it is to further improve recall score and make a predictive model that generalizes even better with unseen data. Depending on their decision, an ML model could be built with increased time complexity using the more time intensive GridSearchCV instead of RandomizedSearchCV as well as utilize a wider range of hyperparameters to see if model performance and generalizability could be further enhanced.

- We could also seek more data from existing/additional sensors (if possible) so that a more generalizable model with even better performance could be built trained on a larger dataset and tested on a larger unseen data